# 1. Data Acquisition

First stage of the pipeline. It reads the raw Colombo Stock Exchange archive and the
EM-DAT disaster export from `data/`, fetches the macroeconomic and global controls
live, and caches four tables to `artifacts/`: `market`, `disasters`, `macro` and
`sp500`. Nothing downstream reads a raw file again.

This is the only slow stage that depends on a network connection. See
[`architecture/data_acquisition.md`](../architecture/data_acquisition.md) for the
Excel layout problem and the source-by-source detail.

## 1.1 Environment and paths

Loads `_shared.py` (paths, the artifact cache helpers, the target definitions) and applies the thesis figure style, then prints the artifact cache so it is visible which upstream stage produced these inputs and when.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "_shared.py").exists() else Path.cwd() / "notebooks"))
from _shared import *  # noqa: F401,F403  paths, artifact cache, target bounds

from src.evaluation import figures as fx
fx.apply_thesis_style()

print("stage inputs available in the artifact cache:")
print(artifact_status().to_string(index=False) if len(artifact_status()) else "  (none yet)")


stage inputs available in the artifact cache:
                     artifact               written_utc rows                                                                                                                                                                                  note
       ablation_block_contrib 2026-09-16T09:56:35+00:00   75                                                                                                                                 Paired-bootstrap contribution of each external block.
              ablation_blocks 2026-09-16T09:56:22+00:00   90                                                                                                                                             Pooled R2/RMSE per external-block config.
       classification_summary 2026-09-16T10:01:08+00:00   24                                                                                                                                     Pre-registered binary labels, po

## 1.2 Source validation and data inventory

Before writing a single line of pipeline code, every claimed data source and external library citation in the project's supporting documents (a "Resource Matrix", a "Detail Guide", and a "Master Expert Panel Blueprint") was checked against reality — either by direct inspection of the files actually supplied, or by live lookups against the source itself. Nothing below is asserted from memory.

### 1.1 Data files actually supplied, as inspected

| File | Usable? | Content | Real date range (verified) | Role |
|---|---|---|---|---|
| `07Market Indices - Daily.xls` (sheet "Index") | ✅ | Daily ASPI, Milanka, S&P SL20 + ~20 sector indices | **2-Jan-1985 → 28-Jun-2023** (5,597 rows ≥2000-01-01) | Primary ASPI price series (Y1) |
| 24 yearly `<year> [Dd]ata.xls[x]` files (2000–2023) | ✅ (23/24) | Per-security daily closing price + volume, **three distinct real report layouts** across the years | 2001 → Q1-2023 volume; 2000 is price-only, no volume field at all | Aggregated market-wide volume (Y2 baseline) |
| `public_emdat_custom_request_2026-02-09_...xlsx` | ✅ | Real EM-DAT export, standard 47-column public schema, 86 disaster records for Sri Lanka | includes the 2025 "Ditwah" storm record | Primary disaster source (exogenous features) |
| `market-capitalization-Oct 12.csv` | ⚠️ supplementary | Single-date company market-cap snapshot | one date only | Not wired into the core pipeline — optional sector-mapping enrichment |
| `2024 ADB Asia SME Monitor - SRI.xlsx` | ⚠️ not relevant | MSME definitions/financing tables | 2019–2023 | **Not** a macro-control source (no GDP/inflation/FX/rate series) — does not fill that gap |
| `CSE All-Share Historical Data.csv` | ❌ dead | Header row only, 55 bytes, zero data rows | — | Discarded |

**Critical, load-bearing finding:** the local archive's real coverage stops at **28-Jun-2023** (price) / **31-Mar-2023** (volume). The thesis's own flagship example event — the **2025 "Ditwah" storm**, shown in the thesis's Fig. 1 — falls entirely outside this window. See §6 below for how this was handled (spoiler: not silently papered over).

### 1.2 Thesis Table 4 (§3.2.2) vs. what this notebook actually uses

| Thesis Table 4 variable | Thesis-named source | What this notebook uses |
|---|---|---|
| ASPI % change (Y1) | "Colombo Stock Exchange (CSE)" | `07Market Indices - Daily.xls` — real CSE-format archive |
| Volume Crash Magnitude (Y2) | CSE | Aggregated per-security volume from the 24 yearly files — same archive |
| Market Recovery Days (Y3) | CSE | Derived from the same ASPI series |
| Disaster Financial Damage / Population Affected / Disaster Type | EM-DAT | Real EM-DAT export, mapped 1:1 |
| Macroeconomic Stability | **"Central Bank of Sri Lanka; World Bank"** (thesis's own words) | World Bank via `wbgapi` — the thesis-named World Bank source; CBSL's higher-frequency series (CCPI/FX/rate) undocumented gap, see §11 |
| Global Market Conditions | **"Yahoo Finance / Bloomberg"** (thesis's own words) | `yfinance`, S&P 500 (`^GSPC`) — thesis-named source, verified live and working |

No source here was substituted from outside what the thesis itself names in Table 4 / §3.3.1.

### 1.3 External citation fact-checks (Resource Matrix / Detail Guide / Blueprint documents)

| Claim | Status | Finding |
|---|---|---|
| `yfinance` ticker `^CSE` exists | ✅ real | Confirmed live: Yahoo lists "COLOMBO IND ALL SHS (^CSE)" |
| `^CSE`'s data is queryable for recent dates | ❌ **false** | Verified via 3 independent methods (yfinance `.download()`, `.history()`, Yahoo's raw chart API directly): the feed stopped updating **~2019** (`regularMarketTime`→Jun-2019, zero rows for any 2020+ range). The ticker *existing as a page* is not the same as it having a *live feed* — an important distinction this notebook's own first pass got wrong before re-checking (see §6). |
| `wbgapi` (World Bank Python package) | ✅ real | `tgherzog/wbgapi`, confirmed working live |
| EM-DAT free academic CSV export | ⚠️ partially correct | Real, but primary export format is **Excel (.xlsx)**, not CSV as claimed; CSV only via the separate EM-VIEW dashboard |
| `nickkunz/smogn`, `paobranco/ImbalancedLearningRegression` | ✅ real | Both confirmed to exist on GitHub |
| `shap` package | ✅ real | Canonical org now `shap/shap` (moved from `slundberg/shap`) |
| **`Bae-Youn/eventstudy`** (cited in two of the supplied documents) | ❌ **fabricated** | No such repository or author exists anywhere. Real package: `LemaireJean-Baptiste/eventstudy` (PyPI `eventstudy`) |
| EM-DAT "Total Damage, Adjusted" column methodology | ✅ verified | doc.emdat.be confirms it applies **OECD CPI** to inflate raw damage relative to Start Year — i.e. it already satisfies a PPP/inflation-adjustment requirement natively (see §5) |

In [2]:
# Paths, RANDOM_STATE and the artifact helpers now come from _shared; this cell only
# reports the environment so the executed notebook records what produced its numbers.
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"repo root : {REPO_ROOT}")
print(f"data dir  : {DATA_DIR}  ({len(list(DATA_DIR.glob('*'))) if DATA_DIR.exists() else 0} files)")
print(f"artifacts : {ARTIFACT_DIR}")
print(f"figures   : {FIGURE_DIR}")
print(f"seed      : {RANDOM_STATE}")


repo root : F:\CSE-disaster-impact-predictor
data dir  : F:\CSE-disaster-impact-predictor\data  (29 files)
artifacts : F:\CSE-disaster-impact-predictor\artifacts
figures   : F:\CSE-disaster-impact-predictor\docs\figures
seed      : 42


## 1.3 Acquiring the raw series

Every loader below reads the **real files described in §1.1**. Each was verified during development by direct inspection of its output against the source workbooks (three distinct per-security report layouts required a header-text-driven parser rather than a fixed-column-index one — column order and even column *count* drift year to year in this real archive; see `src/data_pipeline/cse_raw_loaders.py` docstrings for the full provenance notes).

**Automated test coverage is limited to `feature_eng.py` and `time_aware_smogn.py** (`tests/test_feature_eng.py`, `tests/test_time_aware_smogn.py`). The loaders themselves have no unit tests — recorded here as outstanding work rather than implied to exist.

### 1.3.1 Market series

Parses every yearly per-security workbook and the ASPI index sheet into one daily
`date, aspi_close, trading_volume` series. The loader locates headers by content
because the workbooks use three incompatible layouts across the sample.

In [3]:
from src.data_pipeline.cse_raw_loaders import build_market_dataframe, load_all_yearly_security_files, load_aspi_index
from src.data_pipeline.emdat_loader import load_emdat
from src.data_pipeline.macro_loader import load_worldbank_macro, load_sp500_global_control

# --- Primary CSE market series: real local archive only (see \u00a76 for why no live gap-fill) ---
market = build_market_dataframe(DATA_DIR)[["date", "aspi_close", "trading_volume", "price_source", "volume_source"]]
print(f"Market series: {market.shape[0]} trading days, {market['date'].min().date()} -> {market['date'].max().date()}")
market.head()


F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")


F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")


F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")


F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")
F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")
F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")
F:\CSE-disaster-impact-predictor\src\

F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")
F:\CSE-disaster-impact-predictor\src\data_pipeline\cse_raw_loaders.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], errors="coerce")


[cse_raw_loaders] WARNING: 1 yearly file(s) failed to parse:
  - 2000 data.xls: could not parse any recognizable layout from F:\CSE-disaster-impact-predictor\data\2000 data.xls
Market series: 5597 trading days, 2000-01-03 -> 2023-06-28


,date,aspi_close,trading_volume,price_source,volume_source
0,2000-01-03,574.2,NaN,local_archive,missing
1,2000-01-04,571.2,NaN,local_archive,missing
2,2000-01-05,562.2,NaN,local_archive,missing
3,2000-01-06,556.5,NaN,local_archive,missing
4,2000-01-10,547.7,NaN,local_archive,missing


### 1.3.2 Disaster records

Loads the EM-DAT export and maps its native schema onto the columns the feature
builder expects. Records outside the market series' coverage are loaded and counted
here, then dropped by stage 02's scope filter, so the loss is visible rather than
silent.

In [4]:
# --- Real EM-DAT disaster export ---
# Upgraded 2026-09-12 to the 1986-2025 export (110 records, was 86 covering 2000-2025).
# The extra 24 records are all pre-2000. They are loaded here but CANNOT be modelled: the
# ASPI series starts 2000-01-03, and a day-0 event study needs the index on the event day
# plus 90 trading days after it. Stage 02's scope filter drops them, and the count is
# printed below so the loss is visible rather than silent.
EMDAT_PATH = DATA_DIR / "public_emdat_custom_request_2026-09-12_3d16016a-be39-4159-b9da-35533bf5b9c7.xlsx"
disasters_raw = load_emdat(EMDAT_PATH)
print(f"Raw EM-DAT records: {disasters_raw.shape[0]} "
      f"({disasters_raw['event_date'].min().date()} -> {disasters_raw['event_date'].max().date()})")
print(f"  pre-2000 (no ASPI coverage, cannot be modelled): "
      f"{(disasters_raw['event_date'] < '2000-01-01').sum()}")
print()
print("damage_source breakdown (how much of the set has a real vs. missing damage figure):")
print(disasters_raw["damage_source"].value_counts().to_string())
print()
# EM-DAT leaves Start Day blank for slow-onset events. Filling it with 1 invents a
# first-of-month event date, which for a DAY-0 study means scoring the market on a day the
# disaster did not begin. Recorded, not hidden -- the affected events are droughts, which
# have no meaningful day-0 at all.
print("event-date precision:")
print(disasters_raw["event_date_precision"].value_counts().to_string())
print()
print(f"Magnitude (physical, unit varies by hazard): "
      f"{disasters_raw['mag_area_km2'].notna().sum()} flood-area km2, "
      f"{disasters_raw['mag_wind_kph'].notna().sum()} storm-wind kph")
disasters_raw.head()

Raw EM-DAT records: 110 (1986-01-01 -> 2025-11-27)
  pre-2000 (no ASPI coverage, cannot be modelled): 24

damage_source breakdown (how much of the set has a real vs. missing damage figure):
damage_source
missing_zero_filled    84
emdat_cpi_adjusted     26

event-date precision:
event_date_precision
exact_day     92
month_only    17
year_only      1

Magnitude (physical, unit varies by hazard): 31 flood-area km2, 5 storm-wind kph


,event_date,event_date_precision,date_is_exact,disaster_type,disaster_group,disaster_subgroup,financial_damage,damage_source,financial_damage_observed,population_affected,total_deaths,deaths_available,no_homeless,homeless_available,mag_area_km2,mag_wind_kph,mag_area_available,mag_wind_available,dis_no,event_name
0,1986-01-01,month_only,0.0,Flood,Natural,Hydrological,2743000.0,emdat_cpi_adjusted,1.0,64485.0,43.0,1.0,NaN,0.0,NaN,NaN,0.0,0.0,1986-0178-LKA,NaN
1,1987-01-01,year_only,0.0,Drought,Natural,Climatological,0.0,missing_zero_filled,0.0,2200000.0,NaN,0.0,NaN,0.0,NaN,NaN,0.0,0.0,1987-9025-LKA,NaN
2,1987-11-01,month_only,0.0,Epidemic,Natural,Biological,0.0,missing_zero_filled,0.0,NaN,53.0,1.0,NaN,0.0,NaN,NaN,0.0,0.0,1987-0202-LKA,Encaphalitis
3,1988-08-01,month_only,0.0,Drought,Natural,Climatological,0.0,missing_zero_filled,0.0,806000.0,NaN,0.0,NaN,0.0,NaN,NaN,0.0,0.0,1988-9036-LKA,NaN
4,1989-05-30,exact_day,1.0,Flood,Natural,Hydrological,90895000.0,emdat_cpi_adjusted,1.0,501000.0,325.0,1.0,200000.0,1.0,2735.0,NaN,1.0,0.0,1989-0021-LKA,NaN


### 1.3.3 Macroeconomic controls

Annual World Bank series for Sri Lanka. The dating matters: a year's figures are not
published until the following year, so they are dated accordingly. Dating them to 1
January would hand an event the full-year numbers that summarise the year it occurred
in — look-ahead across four features.

In [5]:
# --- Macro controls: World Bank (real, thesis-named source) ---
macro = load_worldbank_macro(start_year=2000, end_year=2025)
# Annual World Bank figures for year Y are dated Y+1-07-01, not Y-01-01.
# Dating them 1 January made them available to events that occurred DURING the
# year they summarize: a November 2005 event was being handed full-year 2005 GDP
# and inflation, numbers not published until well into 2006. That is look-ahead,
# and it contaminated 4 features (gdp_growth_pct, inflation_cpi_pct,
# gdp_current_usd, and damage_to_gdp which is derived from the last of these).
# Mid-year of the following year approximates the World Bank release schedule.
macro["date"] = pd.to_datetime((macro["year"] + 1).astype(str) + "-07-01")

# Reproducibility caveat, stated rather than implied: this series is pulled LIVE
# from the World Bank API, and its historical values are revised retrospectively.
# The figures below are the revision current as of the run date printed above --
# not the vintage that would have been observable at each event date. Re-running
# this notebook in a later year will produce slightly different macro inputs and
# therefore slightly different model metrics.
print("World Bank macro series (annual):")
macro


World Bank macro series (annual):


series,year,inflation_cpi_pct,gdp_current_usd,gdp_growth_pct,date
0,2000,6.176276,1.659588e+10,6.000033,2001-07-01
1,2001,14.158456,1.574975e+10,-1.545408,2002-07-01
2,2002,9.551032,1.653654e+10,3.964676,2003-07-01
3,2003,6.314638,1.888177e+10,5.940269,2004-07-01
4,2004,7.575926,2.066253e+10,5.445061,2005-07-01
5,2005,11.639686,2.440579e+10,6.241748,2006-07-01
6,2006,10.020184,2.826741e+10,7.668292,2007-07-01
7,2007,15.842111,3.235024e+10,6.796826,2008-07-01
8,2008,22.564496,4.071383e+10,5.950088,2009-07-01
9,2009,3.464963,4.206622e+10,3.538912,2010-07-01


### 1.3.4 Global market control

Daily S&P 500 log returns, the thesis's named control for global market conditions.

In [6]:
# --- Global control: S&P 500 (real, thesis-named source) ---
sp500 = load_sp500_global_control()
print(f"S&P 500 daily log returns: {sp500.shape[0]} rows, {sp500['date'].min().date()} -> {sp500['date'].max().date()}")
sp500.head()


S&P 500 daily log returns: 6537 rows, 2000-01-04 -> 2025-12-30


,date,sp500_log_return
1,2000-01-04,-0.039099
2,2000-01-05,0.001920
3,2000-01-06,0.000955
4,2000-01-07,0.026730
5,2000-01-10,0.011128


## 1.4 Cleaning

**Deviation flagged and corrected (Blueprint Step 1 vs. thesis §3.4.1):** the supplied "Master Expert Panel Blueprint" instructs `.fillna(0)` for missing daily Volume. The thesis explicitly bans this: *"Never use mean-imputation for financial time-series"* and mandates forward-fill only, because the CSE's chronic illiquidity means a missing trading day is not the same as zero volume (§3.4.1, citing Sangeetha & Alfia, 2023). Zero-filling would fabricate a liquidity crash that never happened, corrupting Y2 at the source. This notebook follows the **thesis**, not the blueprint, on this point — the repo's existing `forward_fill_missing_values` already implements it correctly.

In [7]:
from src.data_pipeline.preprocessor import forward_fill_missing_values

market_clean = market.set_index("date")
market_clean = forward_fill_missing_values(market_clean).reset_index()
print(f"Missing aspi_close after ffill: {market_clean['aspi_close'].isna().sum()}")
print(f"Missing trading_volume after ffill: {market_clean['trading_volume'].isna().sum()} "
      f"(real gaps remain where the archive itself has no volume data yet -- 2000, and post Mar-2023)")


Missing aspi_close after ffill: 0
Missing trading_volume after ffill: 241 (real gaps remain where the archive itself has no volume data yet -- 2000, and post Mar-2023)


## 1.5 Extending the ASPI series beyond the archive

The local archive's price series stops at **2023-06-28**, which excluded ten qualifying
EM-DAT events -- including both Storm Ditwah dates, the thesis's own flagship example.
`countryeconomy.com` publishes the daily ASPI close and covers the gap.

The archive stays authoritative for every day it covers: `extend_market_series` appends
only rows strictly after the cutoff, so no existing value is ever overwritten. The 21-day
overlap is checked below as a continuity test rather than assumed.

`trading_volume` is left NaN on the appended rows -- countryeconomy publishes the index
level only, and no practical post-2023 volume source was found (the CSE daily PDFs are
reachable but the index API returns only the five most recent, at ~2.7 MB each). The
existing per-target NaN mask therefore drops these events from Y2, exactly as it already
does for the 2000 volume gap.

In [8]:
from pathlib import Path

from src.data_pipeline.external_sources import extend_market_series, fetch_countryeconomy_aspi

EXTERNAL_CACHE = ARTIFACT_DIR / "external"
EXTERNAL_CACHE.mkdir(parents=True, exist_ok=True)

_archive_end = market_clean["date"].max()
_months = [f"{y}-{m:02d}" for y in range(_archive_end.year, 2027) for m in range(1, 13)]
_months = [m for m in _months if m >= _archive_end.strftime("%Y-%m")]

aspi_ext = fetch_countryeconomy_aspi(EXTERNAL_CACHE, _months)

# Continuity test on the overlap: if the external source disagreed with the archive on
# the days both cover, splicing them would introduce a level break at the join.
_overlap = market_clean.merge(aspi_ext, on="date", suffixes=("_arch", "_ext"))
_diff = (_overlap["aspi_close_arch"] - _overlap["aspi_close_ext"]).abs()
print(f"Overlap days: {len(_overlap)}  |  exact matches: {int((_diff < 0.005).sum())}"
      f"  |  max abs diff: {_diff.max():.2f}  |  mean: {_diff.mean():.3f}")
assert _diff.max() < 100, "External ASPI disagrees with the archive -- do not splice."

market_clean = extend_market_series(market_clean, aspi_ext)
print(f"Market series extended: {_archive_end.date()} -> {market_clean['date'].max().date()} "
      f"(+{(market_clean['date'] > _archive_end).sum()} trading days)")

Overlap days: 21  |  exact matches: 19  |  max abs diff: 42.43  |  mean: 2.097
Market series extended: 2023-06-28 -> 2026-09-11 (+769 trading days)


## 1.6 Cache the acquired data

Everything above is expensive and none of it depends on a modelling choice, so it is
written once and reloaded by every later stage. The provenance sidecar records the
library versions and retrieval timestamp — without it the World Bank and Yahoo pulls are
unpinned and the results table cannot be reproduced later (audit item **P2-8**).

In [9]:
save_frame(market_clean, "market", "Daily ASPI close + summed market volume, forward-filled.")
save_frame(disasters_raw, "disasters", "Raw EM-DAT export, all records before scope filtering.")
save_frame(macro, "macro", "World Bank annual series, publication-lag corrected.")
save_frame(sp500, "sp500", "S&P 500 daily log return, global control.")


cached market.parquet  (6366 rows x 5 cols)


cached disasters.parquet  (110 rows x 20 cols)


cached macro.parquet  (26 rows x 5 cols)


cached sp500.parquet  (6537 rows x 2 cols)


WindowsPath('F:/CSE-disaster-impact-predictor/artifacts/sp500.parquet')